# <font color="orange">FragMOPs Tutorial</font>

This notebook illustrates defining CBU templates, assembling CBUs and MOPs from molecular fragments, and performing genetic optimisation based on the molecular fragments


### Contents:
-	[CBU Templates](#section-1)
-   [Molecular Fragments](#section-2)
-   [Construct CBUs](#section-3)
-   [Assemble MOPs](#section-4)
-   [Genetic Algorithm](#section-5)

In [ ]:
from ontomops import (
    DIRECT_BINDING,
    BindingFragment,
    LinkerFragment,
    NodeFragment,
    MolecularFragment,
    CBUFragmentTemplate,
    ChemicalBuildingUnitTemplate,
    GenericBuildingUnitType,
    AssemblyModel,
    Provenance,
    ChemicalBuildingUnit,
    MetalOrganicPolyhedron,
)

from twa.kg_operations import PySparqlClient
from twa.conf import config_generic, Config

In [ ]:
class MOPsConfig(Config):
    SPARQL_ENDPOINT: str
    SPARQL_USERNAME: str
    SPARQL_PASSWORD: str
    FS_URL: str
    FS_USERNAME: str
    FS_PASSWORD: str
    DATA_DIR: str

ENV_FILE = "./mops.env"
mops_conf = config_generic(MOPsConfig, env_file=ENV_FILE)

# Define SPARQL endpoint for FragMOPs KG, as use these IRIs later
FragMOPs_SPARQL_endpoint = "http://mops.theworldavatar.io:3838/blazegraph/namespace/ontomops_ogm_fragmops/sparql"

sparql_client = PySparqlClient(
    query_endpoint=FragMOPs_SPARQL_endpoint,
    update_endpoint=FragMOPs_SPARQL_endpoint,
    kg_user=mops_conf.SPARQL_USERNAME,
    kg_password=mops_conf.SPARQL_PASSWORD,
    fs_url=mops_conf.FS_URL,
    fs_user=mops_conf.FS_USERNAME,
    fs_pwd=mops_conf.FS_PASSWORD,
)


### 1. Defining CBU templates <a id="section-1"></a>

To create a CBU template we must define a set of template slots. The slots define the fragment type and the positions the fragment will occupy in CBU assembly sequence. We will start by instantiating the fragment types and the GBU type for a simple CBU template corresponding to 2-linear CBUs with one binding group and one cyclic linear linker fragment. 


1. create fragment types



binding types

In [ ]:
co2_binding_type = BindingFragment(
        hasOuterCoordinationNumber=2,
        hasBindingFragment="CO2",
        hasBindingDirection=DIRECT_BINDING,#"direct",
        rdfs_label={"CO2_binding_fragment"}
    )

pyrazole_binding_type = BindingFragment(
    hasOuterCoordinationNumber=2,
    hasBindingFragment="N2",
    hasBindingDirection=DIRECT_BINDING,#"direct",
    rdfs_label={"N2_binding_fragment"}
)

binding_types = [
    co2_binding_type,
    pyrazole_binding_type,
]

Linker types

In [ ]:

linear_cyclic_linker_type = LinkerFragment(
    isLinear=True,
    isCyclic=True,
    rdfs_label={"linear_cyclic_linker_type"}
)

2. Create a GBU type for the CBUs

In [ ]:
two_linear_gbu_type = GenericBuildingUnitType(
    hasModularity=2,
    hasPlanarity="linear",
)

3. Create the CBU Template slots

In [ ]:
binding_slot_0 = CBUFragmentTemplate(
    hasFragmentType=set(binding_types),
    hasFragmentPositions=[0,2], # assembly sequence positions
)

linker_slot_0 = CBUFragmentTemplate(
    hasFragmentType={
        linear_cyclic_linker_type, 
    }, 
    hasFragmentPositions=[1],
)


4. Create the CBU template

In [ ]:
cbu_template1 = ChemicalBuildingUnitTemplate(
    hasGenericBuildingUnitType=two_linear_gbu_type,
    hasCBUFragmentTemplate=[
        binding_slot_0,
        linker_slot_0,
    ],
    rdfs_comment="A 2-linear CBU template with binding group and one cyclic linear linker."
)

We can validate the template to check that there are no overlaping positions

In [ ]:
ChemicalBuildingUnitTemplate.validate_template(cbu_template1)


In general the fragment types and GBU type would be pulled from the knowledge graph if using existing types/templates. Defining new types and templates is only required if does not exist already. The exisiting types and templates can be pulled using the OGM and inspecting the objects comment/label values as shown below.

In [ ]:
binding_types = [
    ft for ft in BindingFragment.pull_all_instances_from_kg(sparql_client, -1)
]
print(f"Found {len(binding_types)} binding frags in KG.")

linker_types = [
    ft for ft in LinkerFragment.pull_all_instances_from_kg(sparql_client, -1)
]
print(f"Found {len(linker_types)} linker frags in KG.")

node_types = [
    ft for ft in NodeFragment.pull_all_instances_from_kg(sparql_client, -1)
]
print(f"Found {len(node_types)} node frags in KG.")

frag_types = {
    list(ft.rdfs_label)[0]: ft 
    for ft in binding_types + linker_types + node_types
} # assuming unique labels

In [ ]:
cbu_templates = ChemicalBuildingUnitTemplate.pull_all_instances_from_kg(sparql_client, -1)
print(f"Found {len(cbu_templates)} CBU templates in KG.")

### 2. Create Fragments <a id="section-2"></a>

Below we demonstrate creating molecular fragments by either reading from a mol file or using SMILES and rdkit 3D conformation embedding


In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem

linker_count = 0
linker_frags = []
linker_mols = []


#### From Mol File

In [ ]:
linker_mol_blocks = []
linker_mol_blocks.append("""o493
     RDKit          2D

 12 13  0  0  0  0  0  0  0  0999 V2000
    2.1325   -0.8264    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.9245   -1.5337    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.1556   -0.6594    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.1556    0.6596    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.8197    0.8515    0.0000 S   0  0  0  0  0  0  0  0  0  0  0  0
    0.8432   -2.6124    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
   -0.9245    1.5339    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -1.8197   -0.8513    0.0000 S   0  0  0  0  0  0  0  0  0  0  0  0
   -2.1325    0.8266    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.8432    2.6125    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    3.1154   -1.2777    0.0000 Er  0  0  0  0  0  1  0  0  0  0  0  0
   -3.1154    1.2779    0.0000 Er  0  0  0  0  0  1  0  0  0  0  0  0
  1  2  2  0
  1  5  1  0
  2  3  1  0
  2  6  1  0
  3  4  2  0
  3  8  1  0
  4  5  1  0
  4  7  1  0
  7  9  2  0
  7 10  1  0
  8  9  1  0
  1 11  1  0
  9 12  1  0
M  END
""")

linker_mol_blocks.append("""o24
     RDKit          3D

 16 20  0  0  0  0  0  0  0  0999 V2000
   -0.7703   -0.7731   -0.7614 C   0  0  2  0  0  0  0  0  0  0  0  0
    0.7737   -0.7731   -0.7614 C   0  0  1  0  0  0  0  0  0  0  0  0
    0.7737    0.7699   -0.7614 C   0  0  1  0  0  0  0  0  0  0  0  0
   -0.7703    0.7699   -0.7614 C   0  0  2  0  0  0  0  0  0  0  0  0
   -1.2223   -1.2261   -1.6674 H   0  0  0  0  0  0  0  0  0  0  0  0
    1.2267   -1.2261   -1.6684 H   0  0  0  0  0  0  0  0  0  0  0  0
    1.2267    1.2229   -1.6684 H   0  0  0  0  0  0  0  0  0  0  0  0
   -0.7703   -0.7731    0.7786 C   0  0  2  0  0  0  0  0  0  0  0  0
   -0.7703    0.7699    0.7786 C   0  0  1  0  0  0  0  0  0  0  0  0
    0.7727    0.7699    0.7786 C   0  0  1  0  0  0  0  0  0  0  0  0
    0.7737   -0.7731    0.7786 C   0  0  2  0  0  0  0  0  0  0  0  0
    1.3907    1.3879    1.3966 H   0  0  0  0  0  0  0  0  0  0  0  0
   -1.3883    1.3879    1.3956 H   0  0  0  0  0  0  0  0  0  0  0  0
   -1.3873   -1.3901    1.3966 H   0  0  0  0  0  0  0  0  0  0  0  0
   -1.0753    1.0759   -1.3744 Er  0  0  0  0  0  1  0  0  0  0  0  0
    1.2067   -1.2061    1.2116 Er  0  0  0  0  0  1  0  0  0  0  0  0
  2  1  1  0
  3  2  1  0
  4  3  1  0
  4  1  1  0
  1  5  1  1
  2  6  1  1
  3  7  1  6
  8  1  1  0
  9  4  1  0
  9  8  1  0
 10  3  1  0
 10  9  1  0
 11  2  1  0
 11  8  1  0
 11 10  1  0
 10 12  1  1
  9 13  1  6
  8 14  1  6
  4 15  1  1
 11 16  1  6
M  END
""")

linker_mol_blocks.append("""o17
     RDKit          3D

 26 29  0  0  0  0  0  0  0  0999 V2000
   -1.7389   -3.0563    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.3459   -3.0563    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.3631   -1.8473    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.3539   -0.6233    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -1.7739   -0.6323    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.4499   -1.8583    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.8021   -1.8053    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.3541    0.6227    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.7731    0.6327    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.4731   -0.6263    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.4501    1.8587    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    3.5501    1.8717    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    1.7391    3.0557    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.3461    3.0557    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.3629    1.8477    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -1.8019    1.8057    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.4729    0.6267    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -3.5729    0.5987   -0.0010 H   0  0  0  0  0  0  0  0  0  0  0  0
   -2.3399    2.7657    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    2.3401   -2.7653   -0.0010 H   0  0  0  0  0  0  0  0  0  0  0  0
    0.2051   -4.0083   -0.0010 H   0  0  0  0  0  0  0  0  0  0  0  0
   -3.5509   -1.8713   -0.0010 H   0  0  0  0  0  0  0  0  0  0  0  0
    3.5731   -0.5983   -0.0010 H   0  0  0  0  0  0  0  0  0  0  0  0
   -0.2049    4.0087    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
   -2.1099   -3.7083   -0.0010 Er  0  0  0  0  0  1  0  0  0  0  0  0
    2.1091    3.7077    0.0000 Er  0  0  0  0  0  1  0  0  0  0  0  0
  2  1  2  0
  3  2  1  0
  4  3  2  0
  5  4  1  0
  6  5  2  0
  6  1  1  0
  7  3  1  0
  8  4  1  0
  9  8  2  0
 10  9  1  0
 10  7  2  0
 11  9  1  0
 12 11  1  0
 13 11  2  0
 14 13  1  0
 15 14  2  0
 15  8  1  0
 16 15  1  0
 17  5  1  0
 17 16  2  0
 18 17  1  0
 19 16  1  0
 20  7  1  0
 21  2  1  0
 22  6  1  0
 23 10  1  0
 24 14  1  0
 25  1  1  0
 26 13  1  0
M  END
""")


linker_mol_blocks.append("""ChemicalBuildingUnit_decce796-ec71-418a-aa3e-a76b32d486a3_1
     RDKit          3D

 18 19  0  0  0  0  0  0  0  0999 V2000
   -2.5530   -0.1256    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.0701   -1.4655    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.7170   -1.7354    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.2391   -0.6786    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.2391    0.6786    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -1.6395    0.9178    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.5530    0.1256    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.0701    1.4655    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.7170    1.7354    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.6395   -0.9178    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    4.0571   -0.1495    0.0000 Er  0  0  0  0  0  1  0  0  0  0  0  0
   -4.0571    0.1495    0.0000 Er  0  0  0  0  0  1  0  0  0  0  0  0
   -2.7879   -2.2991    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
   -0.3718   -2.7798    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
   -2.0076    1.9544    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    2.7879    2.2991    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    0.3718    2.7798    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    2.0076   -1.9544    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
  1  6  2  0
  1  2  1  0
  3  2  2  0
  4  3  1  0
  5  4  2  0
  6  5  1  0
  8  7  1  0
  9  8  2  0
  9  5  1  0
 10  7  2  0
 10  4  1  0
 11  7  1  0
 12  1  1  0
  2 13  1  0
  3 14  1  0
  6 15  1  0
  8 16  1  0
  9 17  1  0
 10 18  1  0
M  END
""")

linker_mol_blocks.append("""o148
     RDKit          3D

 40 46  0  0  0  0  0  0  0  0999 V2000
   -5.0550    1.2710    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -3.5660    1.2390    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.8570    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -3.5660   -1.2390    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -5.0550   -1.2710    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.8750    2.4340    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -1.4370    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.7380    1.2570    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -1.4590    2.4390    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.7380    1.2570    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.4370    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.7380   -1.2570    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.7380   -1.2570    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -1.4590   -2.4390    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.8750   -2.4340    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -3.4210   -3.3860    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
   -0.9470   -3.4060    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
   -3.4210    3.3860    0.0010 H   0  0  0  0  0  0  0  0  0  0  0  0
   -0.9470    3.4060    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    1.4590   -2.4390    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.8570    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.4590    2.4390    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.9470   -3.4060    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    0.9470    3.4060    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    2.8750   -2.4340    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    3.5660   -1.2390    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    2.8750    2.4340    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    3.5660    1.2390    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    3.4210   -3.3860   -0.0010 H   0  0  0  0  0  0  0  0  0  0  0  0
    3.4210    3.3860    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    5.0550   -1.2710   -0.0010 C   0  0  0  0  0  0  0  0  0  0  0  0
    5.0550    1.2710    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    5.7170    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -5.7170    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -5.6000   -2.1920    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
   -5.6000    2.1920    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    5.6000    2.1920    0.0000 H   0  0  0  0  0  0  0  0  0  0  0  0
    5.6000   -2.1920   -0.0020 H   0  0  0  0  0  0  0  0  0  0  0  0
   -6.4670    0.0000    0.0000 Er  0  0  0  0  0  1  0  0  0  0  0  0
    6.4670    0.0000    0.0010 Er  0  0  0  0  0  1  0  0  0  0  0  0
  2  1  2  0
  3  2  1  0
  4  3  2  0
  5  4  1  0
  6  2  1  0
  7  3  1  0
  8  7  2  0
  9  6  2  0
  9  8  1  0
 10  8  1  0
 11 10  2  0
 12 11  1  0
 13 12  2  0
 13  7  1  0
 14 13  1  0
 15 14  2  0
 15  4  1  0
 16 15  1  0
 17 14  1  0
 18  6  1  0
 19  9  1  0
 20 12  1  0
 21 11  1  0
 22 10  1  0
 23 20  1  0
 24 22  1  0
 25 20  2  0
 26 21  2  0
 26 25  1  0
 27 22  2  0
 28 21  1  0
 28 27  1  0
 29 25  1  0
 30 27  1  0
 31 26  1  0
 32 28  2  0
 33 32  1  0
 33 31  2  0
 34  1  1  0
 34  5  2  0
 35  5  1  0
 36  1  1  0
 37 32  1  0
 38 31  1  0
 39 34  1  0
 40 33  1  0
M  END
""")

linker_mol_blocks.append("""o19
     RDKit          2D

  6  5  0  0  0  0  0  0  0  0999 V2000
   -1.9712    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -0.7702    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    0.7698    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
    1.9718    0.0000    0.0000 C   0  0  0  0  0  0  0  0  0  0  0  0
   -2.7212    0.0000    0.0000 Er  0  0  0  0  0  1  0  0  0  0  0  0
    2.7218    0.0000    0.0000 Er  0  0  0  0  0  1  0  0  0  0  0  0
  1  2  3  0
  2  3  1  0
  3  4  3  0
  1  5  1  0
  4  6  1  0
M  END
""")

In [ ]:
for mol_block in linker_mol_blocks:
    linker_count += 1
    fname = f"linker_mol{linker_count}.mol"
    mol = Chem.MolFromMolBlock(mol_block, sanitize=True, removeHs=False)
    Chem.MolToMolFile(mol, fname)
    frag = MolecularFragment.from_mol_file(
        fname,
        fragment_type=linear_cyclic_linker_type,
        dummy_atomic_number=68,
    )
    linker_frags.append(frag)
    linker_mols.append(mol)


#### From SMILES

In [ ]:
def mol_from_smiles(smiles: str) -> Chem.Mol:
    mol = Chem.MolFromSmiles(smiles)
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, AllChem.ETKDG())
    return mol


In [ ]:
# make a CO2 binding fragment
co2_smi = "O=C([O-])[Er]"
co2_mol = mol_from_smiles(co2_smi)
Chem.MolToMolFile(co2_mol, "frag_binding.mol")
co2_frag = MolecularFragment.from_mol_file(
    "frag_binding.mol",
    fragment_type=co2_binding_type,
    dummy_atomic_number=68,
)

In [ ]:
# subsituted benzene linkers
linker_smis = [
    "c1cc([Er])ccc1[Er]",
    "Fc1cc([Er])ccc1[Er]",
    "Clc1cc([Er])ccc1[Er]",
    "Nc1cc([Er])ccc1[Er]",
    "Oc1cc([Er])ccc1[Er]",
]


for smi in linker_smis:
    mol = mol_from_smiles(smi)
    linker_count += 1
    fname = f"frag_linker_{linker_count}.mol"
    Chem.MolToMolFile(mol, fname)
    frag = MolecularFragment.from_mol_file(
        fname,
        fragment_type=linear_cyclic_linker_type,
        dummy_atomic_number=68,
    )
    linker_frags.append(frag)
    linker_mols.append(mol)

In [ ]:
Draw.MolsToGridImage([co2_mol] + linker_mols, molsPerRow=4)

### 3. Assembling CBUs <a id="section-3"></a>

With the fragments and templates in hand, we can now enumerate all valid combinations of the fragments fitting the template. This is done by simply passing the template and fragments to the combinations_from_template_and_fragments function

In [ ]:
import os

# may be required depending on version of ontomops.py
if not os.path.exists("new_cbus"):
    os.mkdir("new_cbus")

In [ ]:
cbus = ChemicalBuildingUnit.combinations_from_template_and_fragments(
    template=cbu_template1,
    fragments=[co2_frag] + linker_frags,
    assemble_smiles=False, # required to be False if assmebling MOPs, if just enumerating space can be True
    sanitize=True,
)

In [ ]:
cbu_mols = [Chem.MolFromSmiles(list(cbu.hasSmiles)[0]) for cbu in cbus]
Draw.MolsToGridImage(cbu_mols, molsPerRow=5)

### 4. Assemble MOPs <a id="section-4"></a>

Can generate individual MOPs from the CBUs using the original MOP assembler routine. It is important that the appropriate GBU is added to the CBU's isFunctioningAs set as shown.

In [ ]:
org_cbu = cbus[0]
org_cbu.hasSmiles

In [ ]:
zr_metal_cbu = ChemicalBuildingUnit.pull_from_kg(
    ['https://www.theworldavatar.com/kg/ontomops/ChemicalBuildingUnit_28a97dd0-d8bd-4f64-8084-687fe8cbd162'],
    sparql_client,
    3
)[0]
zr_metal_cbu.hasCBUFormula

In [ ]:
m4l6_assembly_model = AssemblyModel.pull_from_kg(
    ['https://www.theworldavatar.com/kg/ontomops/AssemblyModel_68aee1b9-3270-4bd3-b6f8-94bde3a74b00'],
    sparql_client,
    3
)[0]
m4l6_assembly_model.rdfs_label

In [ ]:
organic_gbu = next(
    g for g in m4l6_assembly_model.hasGenericBuildingUnit
    if list(g.hasGBUType)[0].label == "2-linear"
)
org_cbu.isFunctioningAs.add(organic_gbu)

In [ ]:
new_mop = MetalOrganicPolyhedron.from_assemble(
    m4l6_assembly_model,
    [org_cbu, zr_metal_cbu],
    prov=Provenance(hasReferenceDOI="Placeholder DOI"),
    sparql_client=sparql_client,
    data_dir=".",
)

In [ ]:
new_mop.hasMOPFormula

In [ ]:
new_mop.visualise()

### 5. Genetic Algorithm Optimisation <a id="section-5"></a>

Here we create a genetic algorithm to demonstrate optimising the MOP properties by searching the CBU space based on fragments. The example illustrates optimising the inner cavity volume to a specified target. However, any MOP property that can be included in the fitness function can be optimised.

The example pulls all available 3-pyramidal metal CBUs and uses the (3-pyramidal)x4(2-linear)x6 assembly model. Combined with the 2-linear cbu template defined previously his creates a gene of:
    [assembly_model][metal cbu][binding group fragment][linker fragment]

The space is then

(1 assembly model) x (7 metal CBUs) x (1 binding group) x (11 linker fragments) = 77 MOPs

Try altering the GA parameters or creating more complex CBU templates to search larger design spaces

In [ ]:
import random
import numpy as np

class GeneticOptimizer:
    """
    A class to run a genetic algorithm, configured directly from a
    ChemicalBuildingUnitTemplate object.
    """

    def __init__(
        self,
        cbu_template,
        all_fragments,
        assembly_models,
        metal_cbus,
        fitness_function,
        pop_size = 30,
        mutation_rate = 0.15,
        tournament_size = 2,
        elitism_percent = 0.05,
        crossover_prob = 1.0,
        remove_duplicates = False,
    ):

        self.cbu_template = cbu_template

        self.fitness_function = fitness_function
        self.pop_size = pop_size
        self.mutation_rate = mutation_rate
        self.elitism_percent = elitism_percent
        self.tournament_size = tournament_size
        self.crossover_prob = crossover_prob
        self.remove_duplicates = remove_duplicates

        self.ordered_slots = sorted(
            list(self.cbu_template.hasCBUFragmentTemplate),
            key=lambda s: min(list(s.hasFragmentPositions))
        )
        
        # prepare lists of valid fragments for each slot
        self.fragment_lists = self._prepare_fragment_lists(all_fragments)

        # set search space for each gene position derived from the prepared lists
        base_bounds = [
            len(assembly_models),
            len(metal_cbus),
            *[len(frags) for frags in self.fragment_lists]
        ]

        # identify linker slots and add direction genes
        direction_bounds = [2 for slot in self.ordered_slots if slot.is_linker_slot]

        # final search space
        self.gene_space_bounds = base_bounds + direction_bounds

        self.population = self._initialize_population()
        self.fitnesses = [float("-inf")] * self.pop_size
        self.best_gene_so_far = None
        self.best_fitness_so_far = float("-inf")
        self.best_aux_data = None


    def _prepare_fragment_lists(self, all_fragments):
        """Creates lists of valid fragments for each slot using the template's logic."""
        fragment_lists = []
        for slot in self.ordered_slots:
            valid_frags = [f for f in all_fragments if slot.accepts(f)]
            
            if not valid_frags:
                raise ValueError(f"No fragments found for slot accepting types: {slot.allowed_types}")
            fragment_lists.append(valid_frags)
        return fragment_lists

    def _initialize_population(self):
        """Initializes a random population based on the gene space."""
        population = []
        for _ in range(self.pop_size):
            gene = [random.randrange(bound) for bound in self.gene_space_bounds]
            population.append(gene)
        return population

    def _mutate(self, gene):
        """Mutates a gene by randomly changing alleles within their bounds."""
        mutated_gene = gene[:]
        for i in range(len(mutated_gene)):
            if random.random() < self.mutation_rate:
                mutated_gene[i] = random.randrange(self.gene_space_bounds[i])
        return mutated_gene

    def _crossover(self, parent1, parent2):
        """Performs single-point crossover between two parents."""
        if len(parent1) < 2:
            return parent1[:], parent2[:]
        pt = random.randrange(1, len(parent1))
        child1 = parent1[:pt] + parent2[pt:]
        child2 = parent2[:pt] + parent1[pt:]
        return child1, child2

    def _tournament_select(self, k = 2):
        """Perform tournament selection based on fitness with size k tournaments"""
        fitness_array = self.fitnesses
        best_idx = -1
        for _ in range(k):
            idx = random.randrange(self.pop_size)
            if best_idx == -1 or fitness_array[idx] > fitness_array[best_idx]:
                best_idx = idx
        return self.population[best_idx][:]


    def run(self, generations, log_fpath = None,):
        """executes the genetic algorithm for the number of generations."""
        generation_history = []

        if log_fpath is not None:
            with open(log_fpath, 'w') as f:
                f.write("generation,individual,cbu_geofile,mop_iri,gene,fitness,volume\n")

        for gen in range(1, generations + 1):
            
            # evaluate population
            volumes = []
            aux_data_list = []
            for i, gene in enumerate(self.population):
                print(f"Evaluating individual {i+1}/{self.pop_size}: {gene}")

                fitness, aux_data = self.fitness_function(
                    gene, self.fragment_lists, self.cbu_template
                )
                self.fitnesses[i] = fitness
                volume = aux_data.pop("volume") if aux_data else None
                volumes.append(volume)
                aux_data_list.append(aux_data)
                if volume is not None:
                     print(f"  Fitness: {fitness:.4f}, Volume: {volume:.1f} Å³")
                
                if log_fpath is not None:
                    with open(log_fpath, 'a') as f:
                        f.write(
                            f"{gen},{i+1},"
                            f"{aux_data.get('cbu_geofile') if aux_data else 'N/A'},"
                            f"{aux_data.get('mop_iri') if aux_data else 'N/A'},"
                            f"\"{gene}\","
                            f"{fitness},"
                            f"{volume if volume is not None else 'N/A'}\n"
                        )

            # population stats
            valid_vols = [v for v in volumes if v is not None]
            pop_avg_vol = float(np.mean(valid_vols)) if valid_vols else ""
            pop_max_vol = float(np.max(valid_vols)) if valid_vols else ""
            pop_min_vol = float(np.min(valid_vols)) if valid_vols else ""
            pop_vol_std = float(np.std(valid_vols)) if valid_vols else ""

            pop_avg_fit = float(np.mean(self.fitnesses))
            pop_max_fit = float(np.max(self.fitnesses))
            pop_min_fit = float(np.min(self.fitnesses))
            pop_fit_std = float(np.std(self.fitnesses))

            # best solution
            gen_best_idx = np.argmax(self.fitnesses)
            gen_best_fitness = self.fitnesses[gen_best_idx]
            gen_best_volume = volumes[gen_best_idx] if volumes else None
            gen_best_data = aux_data_list[gen_best_idx] if aux_data_list else None
            generation_history.append({
                "generation": gen,
                "avg_fitness": pop_avg_fit,
                "max_fitness": pop_max_fit,
                "min_fitness": pop_min_fit,
                "avg_volume": pop_avg_vol,
                "max_volume": pop_max_vol,
                "min_volume": pop_min_vol,
                "fitness_std": pop_fit_std,
                "volume_std": pop_vol_std,
                "best_fitness": gen_best_fitness,
                "best_volume": gen_best_volume,
                "best_gene": self.population[gen_best_idx][:],
                **gen_best_data
            })

            if self.fitnesses[gen_best_idx] > self.best_fitness_so_far:
                self.best_fitness_so_far = self.fitnesses[gen_best_idx]
                self.best_gene_so_far = self.population[gen_best_idx][:]
                self.best_aux_data = {"volume": volumes[gen_best_idx]}
                self.best_aux_data.update(aux_data_list[gen_best_idx] if aux_data_list else {})

            
            # create next generation
            new_pop = []
            
            # elitism
            sorted_indices = np.argsort(self.fitnesses)[::-1]
            elitism_count = max(2, int(self.elitism_percent * self.pop_size))
            for i in range(elitism_count):
                new_pop.append(self.population[sorted_indices[i]][:])


            while len(new_pop) < self.pop_size:
                p1 = self._tournament_select(k=self.tournament_size)
                p2 = self._tournament_select(k=self.tournament_size)

                if random.random() < self.crossover_prob:
                    c1, c2 = self._crossover(p1, p2)
                else:
                    # no crossover
                    c1, c2 = p1[:], p2[:]

                new_pop.append(self._mutate(c1))
                if len(new_pop) < self.pop_size:
                    new_pop.append(self._mutate(c2))
            
            # Remove duplicate genes and replace with random genes
            if self.remove_duplicates:
                new_pop = [list(x) for x in set(tuple(x) for x in new_pop)]
                new_pop += self._initialize_population()[0:self.pop_size - len(new_pop)]
            
            self.population = new_pop

        return self.best_gene_so_far, self.best_fitness_so_far, self.best_aux_data, generation_history


In [ ]:
def assemble_cbu_from_gene(
    gene,
    fragment_lists,
    cbu_template,
    gbu,
    gbu_type,
    sanitize,
    optimize = False
):
    """
    assembles a CBU from a gene
    """
    num_fragment_slots = len(fragment_lists)
    fragment_indices = gene[2 : 2 + num_fragment_slots]
    direction_alleles = gene[2 + num_fragment_slots:]

    frags = [
        fragment_lists[i][fragment_indices[i]] for i in range(num_fragment_slots)
    ]

    linker_directions = []
    direction_allele_counter = 0
    ordered_slots = sorted(
        list(cbu_template.hasCBUFragmentTemplate),
        key=lambda s: min(list(s.hasFragmentPositions))
    )
    
    for slot in ordered_slots:
        if slot.is_linker_slot:
            linker_directions.append(direction_alleles[direction_allele_counter])
            direction_allele_counter += 1

    cbu = ChemicalBuildingUnit.create_cbu_from_ordered_fragments_and_template(
        template=cbu_template,
        fragments=frags,
        gbu=gbu,
        sanitize=sanitize,
        optimize=optimize,
        is_2bent=("bent" in gbu_type.label),
        linker_first_dummy_idxs=linker_directions,
        symmetric_linker_orientations=True,
    )
    return cbu  



In [ ]:
from rdkit.Contrib.SA_Score import sascorer
import math
import os

MOPS_DIRECTORY = "./mops_data"

if not os.path.exists(MOPS_DIRECTORY):
    os.makedirs(MOPS_DIRECTORY)


def evaluate_mop_fitness(
    gene,
    fragment_lists,
    cbu_template,
    target_vol,
    sanitize,
    assembly_models,
    metal_cbus,
    prov,
    sparql_client,
    optimize = False,
):
    """
    build a MOP from gene and evaluate the fitness
    """
    try:
        am_idx, metal_idx = gene[0], gene[1]
        am = assembly_models[am_idx]
        metal_cbu = metal_cbus[metal_idx]
        gbu_type_label = cbu_template.gbu_type

        gbu = next(
            (x for x in am.hasGenericBuildingUnit if list(x.hasGBUType)[0].label == gbu_type_label),
            None
        )

        gbu_type = list(gbu.hasGBUType)[0]

        org_cbu = assemble_cbu_from_gene(
            gene, fragment_lists, cbu_template, gbu, gbu_type, sanitize, optimize=optimize
        )

        new_mop = MetalOrganicPolyhedron.from_assemble(
            am,
            [org_cbu, metal_cbu],
            prov,
            sparql_client=sparql_client,
            data_dir=MOPS_DIRECTORY,
        )

        mol = Chem.MolFromSmiles(list(org_cbu.hasSmiles)[0])
        sa_score = sascorer.calculateScore(mol)

        # Extract volume
        cavity = list(new_mop.hasCavity)[0]
        inner_sphere = list(cavity.hasLargestInnerSphereDiameter)[0]
        inner_sphere_val = list(inner_sphere.hasValue)[0]
        inner_sphere_val_literal = list(inner_sphere_val.hasNumericalValue)[0]
        volume =  (4.0/3.0) * math.pi * (inner_sphere_val_literal/2) ** 3

        # calculate number of sidechain fragments
        cbu_frags = list(org_cbu.hasChemicalBuildingUnitFragment)
        n_sidechain_frags = 0
        for cbu_frag in cbu_frags:
            _frag = list(cbu_frag.hasMolecularFragment)[0]
            if _frag.hasSideChainFragments is None:
                continue
            _n_positions = len(list(cbu_frag.hasFragmentPositions))
            n_sidechain_frags += len(list(_frag.hasSideChainFragments))*_n_positions


        # calculate fitness
        fitness = -abs(volume - target_vol) - (10 * sa_score) - (10 * n_sidechain_frags)

        aux_data = {
            "volume": volume,
            "sa_score": sa_score,
            "cbu_smiles": list(org_cbu.hasSmiles)[0],
            "mop_iri": new_mop.instance_iri,
            "cbu_iri": org_cbu.instance_iri,
            "cbu_geofile": list(list(org_cbu.hasGeometry)[0].hasGeometryFile)[0]
        }

        return fitness, aux_data

    except Exception as e:
        print(f"Failed to evaluate gene {gene}: {e}")
        return -1e6, {"volume": None}


In [ ]:
random.seed(123)
np.random.seed(123)

prov = Provenance(hasReferenceDOI="placeholder doi")

ga_params = {
    'target_volume': 625,
    'pop_size': 30,
    'generations': 10,
    'sanitize': True,
    'prov': prov,
    'sparql_client': sparql_client,
    'optimize': False,
}


In [ ]:
metal_cbus = [
    cbu for cbu in ChemicalBuildingUnit.pull_all_instances_from_kg(sparql_client, 3)
    if cbu.is_metal_cbu and list(cbu.isFunctioningAs)[0].gbu_type == "3-pyramidal"
]

In [ ]:
from functools import partial

# create the fitness function
fitness_func = partial(
    evaluate_mop_fitness,
    target_vol=ga_params['target_volume'],
    sanitize=ga_params['sanitize'],
    optimize=ga_params['optimize'],
    assembly_models=[m4l6_assembly_model], 
    metal_cbus=metal_cbus,
    prov=ga_params['prov'], 
    sparql_client=ga_params['sparql_client'],
)

In [ ]:
optimizer = GeneticOptimizer(
    cbu_template=cbu_template1,
    all_fragments=linker_frags + [co2_frag],
    assembly_models=[m4l6_assembly_model],
    metal_cbus=metal_cbus,
    fitness_function=fitness_func,
    pop_size=ga_params['pop_size'],
)

In [ ]:
best_gene, best_fit, best_data, generation_history = optimizer.run(
    ga_params['generations'],
)

In [ ]:
best_data

In [ ]:
best_cbu = assemble_cbu_from_gene(
    best_gene,
    optimizer.fragment_lists,
    cbu_template1,
    gbu=organic_gbu,
    gbu_type=list(organic_gbu.hasGBUType)[0],
    sanitize=ga_params['sanitize'],
)

In [ ]:
best_metal_cbu = metal_cbus[best_gene[1]]

In [ ]:
best_mop = MetalOrganicPolyhedron.from_assemble(
    m4l6_assembly_model,
    [best_cbu, best_metal_cbu],
    prov=prov,
    sparql_client=sparql_client,
    data_dir=".",
)

In [ ]:
best_mop.visualise()

In [ ]:
import plotly.graph_objs as go

generation_nums = [h["generation"] for h in generation_history]
best_volume = [h["best_volume"] for h in generation_history]
avg_volume = [h["avg_volume"] for h in generation_history]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=generation_nums,
    y=best_volume,
    mode='lines+markers',
    name='Best',
    line=dict(
        color="#000000",
        dash='dashdot',
    )
))
fig.add_trace(go.Scatter(
    x=generation_nums,
    y=avg_volume,
    mode='lines+markers',
    name='Population Avg.',
    line=dict(
        color="grey",
        dash='dashdot',
    )
))

# add target volume line
fig.add_shape(
    type='line',
    x0=min(generation_nums)-1,
    y0=ga_params['target_volume'],
    x1=max(generation_nums)+1,
    y1=ga_params['target_volume'],
    line=dict(
        color="#D55E00",
    ),
    name='Target Volume'
)

fig.add_trace(go.Scatter(
    x=[None],
    y=[None],
    mode='lines',
    line=dict(
        color="#D55E00",
    ),
    name='Target Volume'
))

fig.update_layout(
    title='Genetic Algorithm',
    xaxis_title='Generation',
    yaxis_title=r"Volume (Ang^3)"
)

fig.show()